# 05 — Portfolio Analytics

Combines individual symbol results into a portfolio NAV and measures risk-adjusted
performance.  Three capital models are compared — rebalanced, buy-and-hold, and
fixed-stake — alongside standard metrics (Sharpe, Calmar, drawdown).

**Pipeline**
```
YahooFinanceProvider  →  TrendSignal  →  BarBacktest  →  SignalAnalytics
      (bars)            (signal cols)   (bt_result)       (portfolio_equity / metrics)
```

In [ ]:
import pandas as pd
import plotly.graph_objects as go

from hailmary.data.providers import YahooFinanceProvider
from hailmary.models import TrendSignal
from hailmary.backtest.signal_backtest import BarBacktest
from hailmary.analytics.signal_analytics import SignalAnalytics
from hailmary.viz.theme import PALETTE, apply_theme

## 1. Data → Signal → Backtest

In [ ]:
signal = TrendSignal(ma_window=200)
yahoo  = YahooFinanceProvider()
symbols = ["BTC-USD", "ETH-USD", "SOL-USD"]
start, end = pd.Timestamp("2022-01-01"), pd.Timestamp("2024-01-01")

fetch_start = start - pd.offsets.BDay(signal.warmup)
bars = yahoo.get_bars(symbols, start=fetch_start, end=end, adjust=False)

signal_df = signal.run(bars, trim_start=start)
bt_result = BarBacktest().run(signal_df)
analytics = SignalAnalytics(bt_result)

colours      = [PALETTE["accent_blue"], PALETTE["accent_green"], PALETTE["accent_purple"]]
fill_colours = ["rgba(88,166,255,0.12)", "rgba(63,185,80,0.12)", "rgba(163,113,247,0.12)"]

eq_con = analytics.equity(method="conservative")

print(f"{bt_result.data.shape[0]:,} bars  |  {bt_result.data.index.get_level_values('symbol').nunique()} symbols")

## 2. Drawdown

Per-symbol drawdown from the conservative equity curve.  Flat periods reflect the signal
being off — no capital is deployed so the NAV holds still and drawdown does not accumulate.

In [ ]:
dd = eq_con / eq_con.cummax() - 1

fig = go.Figure()
for i, sym in enumerate(dd.columns):
    fig.add_trace(go.Scatter(
        x=dd.index, y=dd[sym],
        name=sym,
        fill="tozeroy",
        line=dict(color=colours[i % len(colours)], width=1),
        fillcolor=fill_colours[i % len(fill_colours)],
    ))

apply_theme(fig, title="Drawdown — MA-200 Trend Signal (Conservative)", height=380)
fig.update_layout(yaxis_title="Drawdown", yaxis_tickformat=".0%")
fig.show()

## 3. Trade Duration Distribution

How many bars does each trade last?  A tight distribution close to the median suggests
the signal consistently holds positions for similar durations; a long right tail indicates
a few very extended trades that may be distorting cumulative returns.

In [ ]:
df = bt_result.data

trade_durations = (
    df[df["cycle"] != "None"]
    .reset_index(level="symbol")
    .groupby(["symbol", "trade_cycle_id"])
    .size()
    .reset_index(name="duration_bars")
)

fig = go.Figure()
for i, sym in enumerate(trade_durations["symbol"].unique()):
    durations = trade_durations.loc[trade_durations["symbol"] == sym, "duration_bars"]
    fig.add_trace(go.Histogram(
        x=durations,
        name=sym,
        marker_color=colours[i % len(colours)],
        opacity=0.75,
        nbinsx=20,
    ))

apply_theme(fig, title="Trade Duration Distribution (bars held)", height=380)
fig.update_layout(
    xaxis_title="Duration (bars)",
    yaxis_title="Count",
    barmode="overlay",
)
fig.show()

## 4. Combined Portfolio — Equal-Weight

`portfolio_equity()` weights each in-signal symbol at `1 / N_universe` at each bar —
uninvested capital sits as cash when fewer symbols are in signal.  The bold yellow line
is the combined portfolio; muted dotted lines are individual symbols.

In [ ]:
port_eq = analytics.portfolio_equity(method="conservative")

fig = go.Figure()

for i, sym in enumerate(eq_con.columns):
    fig.add_trace(go.Scatter(
        x=eq_con.index, y=eq_con[sym],
        name=sym,
        line=dict(color=colours[i % len(colours)], width=1, dash="dot"),
        opacity=0.5,
    ))

fig.add_trace(go.Scatter(
    x=port_eq.index, y=port_eq,
    name="Equal-weight Portfolio",
    line=dict(color=PALETTE["accent_yellow"], width=2.5),
))

apply_theme(fig, title="Equal-Weight Portfolio vs Individual Symbols (Conservative)", height=460)
fig.update_layout(yaxis_title="Equity (normalised to 1.0)")
fig.show()

## 5. Capital Models — Side-by-Side

Three ways to aggregate individual symbol returns into a portfolio NAV:

| Model | Within a trade | Between trades | Use when |
|---|---|---|---|
| **`rebalanced`** | Daily reset to `1/N_universe` weight | Geometric | Comparing signal quality independent of capital size |
| **`buy_and_hold`** | Compounds from `1/N_universe` entry | Geometric (winners grow) | Long-only momentum with no rebalancing |
| **`fixed_stake`** | Compounds from `$amount_per_entry` | Arithmetic (P&L to cash) | Fixed-size position sizing |

All three curves start at 1.0 (NAV divided by `amount × N_universe`) so they are directly comparable.

In [ ]:
port_rebalanced = analytics.portfolio_equity(method="conservative", mode="rebalanced")
port_buy_hold   = analytics.portfolio_equity(method="conservative", mode="buy_and_hold")
port_fixed      = analytics.portfolio_equity(method="conservative", mode="fixed_stake", amount_per_entry=1_000)

fig = go.Figure()

for i, sym in enumerate(eq_con.columns):
    fig.add_trace(go.Scatter(
        x=eq_con.index, y=eq_con[sym],
        name=sym, showlegend=True,
        line=dict(color=colours[i % len(colours)], width=1, dash="dot"),
        opacity=0.35,
    ))

fig.add_trace(go.Scatter(
    x=port_rebalanced.index, y=port_rebalanced,
    name="Rebalanced (1/N daily)",
    line=dict(color=PALETTE["accent_yellow"], width=2, dash="dash"),
))
fig.add_trace(go.Scatter(
    x=port_buy_hold.index, y=port_buy_hold,
    name="Buy & Hold (let it run)",
    line=dict(color=PALETTE["accent_green"], width=2),
))
fig.add_trace(go.Scatter(
    x=port_fixed.index, y=port_fixed,
    name="Fixed $1k/entry",
    line=dict(color=PALETTE["accent_blue"], width=2.5),
))

apply_theme(fig, title="Portfolio Capital Models — Conservative Fill", height=500)
fig.update_layout(yaxis_title="NAV (normalised to 1.0)")
fig.show()

## 6. Performance Metrics

Risk-adjusted metrics for each capital model using conservative fill.  Sharpe and
Calmar are the primary ranking metrics; max drawdown sets the risk budget ceiling.

In [ ]:
def _metrics_row(pm) -> dict:
    return {
        "Total Return": f"{pm.total_return:+.1%}",
        "CAGR":         f"{pm.annualised_return:+.1%}",
        "Ann. Vol":     f"{pm.annualised_vol:.1%}",
        "Sharpe":       f"{pm.sharpe:.2f}",
        "Sortino":      f"{pm.sortino:.2f}",
        "Max Drawdown": f"{pm.max_drawdown:.1%}",
        "Calmar":       f"{pm.calmar:.2f}",
    }


rows = {
    "Rebalanced":       _metrics_row(analytics.portfolio_metrics(method="conservative", mode="rebalanced")),
    "Buy & Hold":       _metrics_row(analytics.portfolio_metrics(method="conservative", mode="buy_and_hold")),
    "Fixed $1k/entry":  _metrics_row(analytics.portfolio_metrics(method="conservative", mode="fixed_stake", amount_per_entry=1_000)),
}

pd.DataFrame(rows).T

## 7. Full Metrics — Rebalanced Model

Complete `PerformanceMetrics` summary for the rebalanced portfolio (conservative fill).
Includes VaR, CVaR, skewness, kurtosis, and Omega ratio.

In [ ]:
analytics.portfolio_metrics(method="conservative", mode="rebalanced").summary()